# Network Structure and Nonlinear Price Adjustment in India's Wholesale Onion Market
## Replication notebook

This notebook runs every computation in the paper, from the raw AGMARKNET file to the numbers quoted in the text,
in the order of `code/run_all.py`. Each step runs one script of the replication package and shows its output.
Run all cells from top to bottom (about 20–40 minutes on a laptop). Place
`onion_prices_master_2021_2024_corrected.xlsx` in `data/raw/` first.

In [ ]:
import os, pandas as pd
from IPython.display import Image, Markdown, display
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.join("..", "code"))
print("working directory:", os.getcwd())
def show(name):
    p = os.path.join("..", "output", "tables", name + ".md")
    if os.path.exists(p):
        display(Markdown(open(p, encoding="utf8").read()))

### Step 1. `01_clean_data.py` 

*Feeds:* Section IV.A; Table 2

Load AGMARKNET master file, clean, and select the 240-mandi analysis sample.  
  
Corrections relative to the original notebooks (v5-v8):  
  * Mandis are identified by (State, Market). `Market` alone is not unique: nine market  
    names occur in two states (e.g. Sirsa, Siwan, Pratapgarh) and were previously merged  
    into a single price series.  
  * Records with a modal price above 5x or below 1/5 of the same-day national median are  
    dropped (unit errors such as Rs 0.8 or Rs 460,000 per quintal).  
  * Duplicate mandi-day records are collapsed (arrivals summed, log modal price  
    arrival-weighted).  
Outputs: data/derived/master_clean.parquet, data/derived/panel240.parquet, results['data'].

In [ ]:
%run 01_clean_data.py
show('table1_data_summary')

### Step 2. `00_geocode_check.py` 

*Feeds:* Section IV.A; Appendix C; Figure 1

Check and correct mandi coordinates against an offline gazetteer (record level).  
  
The source file attaches coordinates to every record, and the same mandi often appears with  
several coordinate pairs (1,174 of 1,546 mandis have more than one). Some are far from the mandi's  
state: Anand (Gujarat) has 823 records at 22.47N 72.74E and 270 records at 27.10N 83.27E, a point in  
eastern Uttar Pradesh. This matters beyond maps: the file's daily rainfall is read from the IMD  
grid AT the record's coordinates, so a misplaced record carries rainfall from the wrong location.  
  
Gazetteer: GeoNames cities1000, Indian places (3,435 places with state), as bundled with the  
reverse_geocoder 1.5.1 package (data/raw/geonames_cities1000_india.csv, the Indian places of the worldwide file; GeoNames, CC-BY 4.0).  
  
Procedure  
 1. Coordinate variants: each mandi's distinct (lat, lon) pairs rounded to 0.01 deg.  
 2. A variant is IN-STATE if the nearest gazetteer place in the mandi's stated state is <= 100 km  
    away (tolerates mandis near state borders).  
 3. Canonical location = the in-state variant with the most records. If no variant is in-state,  
    the mandi is relocated to a gazetteer place in the stated state matching the market name, else  
    the district name, else the median location of other mandis in the same state and district.  
 4. A record is FLAGGED if its variant is out of state or lies > 50 km from the canonical location.  
    Flagged records keep their prices but their rainfall is not used (03_rainfall.py).  
Outputs: data/derived/mandi_coordinates.csv (canonical location per mandi),  
data/derived/record_location_flags.parquet, results['geocode'].

In [ ]:
%run 00_geocode_check.py

### Step 3. `verify_original.py` 

*Feeds:* Appendix E

Reproduction check: re-run the ORIGINAL v5 network pipeline (no cleaning, `Market` as the  
mandi key) to confirm that this package reproduces the numbers reported in the original  
notebooks (240 nodes, 714 edges, Fiedler 0.039067, six communities of 58/47/47/38/25/25,  
48 SIMAMs). Any difference between the paper and the corrected results is therefore due to  
the data corrections, not to re-implementation.

In [ ]:
%run verify_original.py

### Step 4. `02_network.py` 

*Feeds:* Section V.A; Tables 3–4

Ledoit-Wolf correlations, PMFG, centrality, SIMAMs, Louvain communities, spectra.  
  
Design follows the original v5 notebook (returns between consecutive reports of a mandi;  
missing returns set to the mandi's mean return; Ledoit-Wolf shrinkage; PMFG with  
d_ij = sqrt(2(1-rho_ij)) and edge weight 1/d_ij), applied to the cleaned panel.  
Usage: python 02_network.py [--shrinkage 0.15] [--tag alpha015]   (robustness variants)

In [ ]:
%run 02_network.py
show('table_communities')

### Step 5. `03_rainfall.py` 

*Feeds:* Section IV.A.2

Monthly rainfall anomalies relative to 1991-2020 normals, at mandi grid cells; community exposure.  
  
Observed rainfall. The AGMARKNET file carries IMD RF25 (0.25 deg) daily rainfall at each record's  
coordinates on trading dates. Records whose coordinates are out of state or > 50 km from the mandi's  
verified location (00_geocode_check.py) are dropped. Each remaining record is assigned to its RF25  
grid cell. For each cell and month, observed rainfall = mean daily value over observed dates x days  
in month; cell-months with fewer than half of days observed are not used.  
  
Normals. 1991-2020 (WMO/IMD standard 30-year period) monthly mean and SD at each of the same 822  
grid cells, computed from the RF25 NetCDF files by notebooks/RF25_normals_1991_2020_mandi_cells.ipynb  
(data/raw/rf25_cell_month_normals_1991_2020.xlsx).  
  
Subdivision-month measures average the cell-level quantities over the cells observed that month, so  
the observations and the baseline always refer to the same locations:  
  rain_mm, normal_mm   mean over cells  
  deficit_mm           max(0, normal_mm - rain_mm)  
  excess_mm            max(0, rain_mm - normal_mm)  
  pdef                 max(0, -(rain_mm/normal_mm - 1))  (IMD-style percentage departure; normal >= 10 mm)  
  z, zdef, absz        mean over cells of (R - mean)/SD, its negative part, its absolute value  
Robustness baseline that uses no external normals: 2021-24 subdivision-month mean and SD (deficit_ins, zdef_ins).  
Previous versions used lpa_normals_2010_2020.csv, which the author's notebook shows was built from 1990-1999  
files at one hand-placed grid cell per subdivision; it is no longer used (comparison saved for the log).  
Requires 02_network.py (community labels).

In [ ]:
%run 03_rainfall.py

### Step 6. `04_indices_cointegration.py` 

*Feeds:* Section V.B; Appendix Tables B1–B2

Community price indices, unit roots, VAR lag order, Johansen and Engle-Granger tests.  
  
Main index: arrival-weighted mean of log modal prices across the community's mandis that  
report on a date (as in the original code; the paper text previously said "equal-weighted").  
Robustness index: two-way fixed-effects (time-dummy) index that removes mandi level  
differences, so that day-to-day changes in which mandis report do not move the index.  
Engle-Granger direction: for pair (i, j) with i < j, C_j is regressed on C_i, i.e. the  
community with the smaller arrival share on the one with the larger share (C0 has the  
largest share). P-values use MacKinnon residual-based critical values (statsmodels.coint).

In [ ]:
%run 04_indices_cointegration.py
show('table_unit_roots')
show('table_engle_granger')

### Step 7. `04b_johansen_cases.py` 

*Feeds:* Section 4.2 (Johansen deterministic cases)

Johansen trace statistics under the four standard deterministic cases (Section 4.2).  
  
statsmodels' coint_johansen covers an unrestricted constant (det_order=0) and an unrestricted trend (det_order=1).  
This script computes the reduced-rank regression directly so that the restricted-constant and restricted-trend  
cases can be compared as well. 5 percent critical values for the last rank (r <= K-1) are those of MacKinnon, Haug  
and Michelis (1999): 3.84 (unrestricted constant or trend), 9.16 (restricted constant), 12.52 (restricted trend).

In [ ]:
%run 04b_johansen_cases.py

### Step 8. `05_tvecm.py` 

*Feeds:* Section V.C–D; Tables 5–6

Two-regime threshold VECM for every community pair, bootstrap threshold tests,  
regime-equality tests, Band-TAR asymmetry, and robustness variants.  
  
Specification (Hansen and Seo 2002, pairwise, symmetric threshold on |ECT|):  
  Y_t = [dC_i,t, dC_j,t]'  
  Y_t = A_k' X_t + e_t,   k = 1 if |ECT_{t-1}| <= gamma (inner regime), k = 2 otherwise (outer regime)  
  X_t = [1, ECT_{t-1}, dC_i,t-1..t-L, dC_j,t-1..t-L, 11 month dummies],  L = VAR(BIC) order - 1 = 2  
gamma minimises log det Sigma(gamma) over 60 points between the 15th and 85th percentiles of |ECT|.  
Adjustment of the disequilibrium itself: dECT_t = dC_j - beta dC_i, so its coefficient on ECT_{t-1}  
in regime k is theta_k = rho_j,k - beta * rho_i,k (identical to OLS of dECT on X_t within regime k).  
Inference: sup-LR threshold test with fixed-regressor residual bootstrap (B = 999); H0: theta_1 =  
theta_2 by Newey-West (10 lags) Wald test conditional on gamma-hat.  
Usage: python 05_tvecm.py            (main + robustness)

In [ ]:
%run 05_tvecm.py
show('table_tvecm_main')

### Step 9. `06_rainfall_moderation.py` 

*Feeds:* Section V.E; Table 7

Does rainfall stress widen inter-community price disequilibrium?  
  
Pair-month panel: |ECT_ij| averaged within month (28 pairs x 48 months).  
Exposure of a pair: mean of the two communities' arrival-weighted deficits (mm), or the  
absolute DIFFERENCE in the two communities' standardised anomalies (a common shock should  
not move relative prices; a differential shock should).  
Specifications: (1) pair FE (as in the original notebook, HC3);  
(2) pair FE + calendar-month FE; (3) pair FE + year-month FE. (2)-(3) cluster by year-month.

In [ ]:
%run 06_rainfall_moderation.py
show('table_rain_moderation')

### Step 10. `07_fiedler_rolling.py` 

*Feeds:* Section V.E; Appendix Figure B1

Time-varying algebraic connectivity and rainfall.  
  
Rolling 90-trading-day Ledoit-Wolf correlations re-weight the edges of the pooled PMFG  
(weight = max(rho_t, 0.001)), and the Fiedler value of the weighted Laplacian is recorded every  
7 trading days. Main version uses all 240 mandis; the original notebook used the top-60 mandis by  
arrivals (reported for comparison). Monthly averages are regressed on the arrival-weighted system  
deficit with Newey-West (3 lags) standard errors, with and without calendar-month dummies.  
Note: overlapping 90-day windows mean that each monthly Fiedler value reflects about three months  
of price data, so the regression is a reduced-form association, not a causal estimate.

In [ ]:
%run 07_fiedler_rolling.py
show('table_fiedler_rain')

### Step 11. `08_rain_price_timing.py` 

*Feeds:* Section V.E; Figure 3

Timing of rainfall effects on community price levels (monthly local projections).  
  
The original notebook regressed h-day forward cumulative returns on a monthly deficit that is  
constant within the month, using overlapping daily observations and HC3 errors, and called this a  
Granger-causality horizon test. Because the deficit is observed monthly, horizons shorter than a  
month are not identified, overlapping windows invalidate HC3 inference, and without calendar-month  
controls the deficit proxies for the seasonal price cycle. Here:  
  y_{c,m+h} - y_{c,m-1} = b_h * shock_{c,m} + community FE + calendar-month FE + e,  h = 0..3  
with y the monthly mean of the daily community log-price index and standard errors clustered by  
year-month. A version without calendar-month FE is reported to show the seasonal confound.

In [ ]:
%run 08_rain_price_timing.py
show('table_rain_price_timing')

### Step 12. `09_policy_bounds.py` 

*Feeds:* Section VIII.C; Table 8

Buffer-stock trigger accounting (illustrative upper bounds).  
  
What the original notebook computed: on days the system price index exceeds a trigger, price is  
set equal to the trigger; "volatility reduction" is the fall in the SD of this truncated series and  
"consumer surplus" is sum((P - trigger) x arrivals). The "optimal" trigger was the lowest grid  
trigger with activation frequency below 20%, i.e. approximately the 80th percentile by construction.  
Two errors are corrected: arrivals are in tonnes and prices in Rs/quintal (1 t = 10 q), so the  
rectangle must be multiplied by 10; and the quantity is a transfer to buyers at sampled APMCs if  
releases could hold the price at the trigger, not a welfare gain. We report: the transfer upper  
bound, the efficiency (Harberger) triangle under a demand elasticity e, and the release volume  
needed to achieve the cap, dQ = |e| * (P - trigger)/P * Q.

In [ ]:
%run 09_policy_bounds.py
show('table_policy_bounds')

### Step 13. `02_network.py` --shrinkage 0.15 --tag alpha015

*Feeds:* Section V.A (partition robustness)

Ledoit-Wolf correlations, PMFG, centrality, SIMAMs, Louvain communities, spectra.  
  
Design follows the original v5 notebook (returns between consecutive reports of a mandi;  
missing returns set to the mandi's mean return; Ledoit-Wolf shrinkage; PMFG with  
d_ij = sqrt(2(1-rho_ij)) and edge weight 1/d_ij), applied to the cleaned panel.  
Usage: python 02_network.py [--shrinkage 0.15] [--tag alpha015]   (robustness variants)

In [ ]:
%run 02_network.py --shrinkage 0.15 --tag alpha015

### Step 14. `02_network.py` --shrinkage 0.50 --tag alpha050

*Feeds:* Section V.A (partition robustness)

Ledoit-Wolf correlations, PMFG, centrality, SIMAMs, Louvain communities, spectra.  
  
Design follows the original v5 notebook (returns between consecutive reports of a mandi;  
missing returns set to the mandi's mean return; Ledoit-Wolf shrinkage; PMFG with  
d_ij = sqrt(2(1-rho_ij)) and edge weight 1/d_ij), applied to the cleaned panel.  
Usage: python 02_network.py [--shrinkage 0.15] [--tag alpha015]   (robustness variants)

In [ ]:
%run 02_network.py --shrinkage 0.50 --tag alpha050

### Step 15. `02_network.py` --pairwise --tag pairwise

*Feeds:* Section V.A (partition robustness)

Ledoit-Wolf correlations, PMFG, centrality, SIMAMs, Louvain communities, spectra.  
  
Design follows the original v5 notebook (returns between consecutive reports of a mandi;  
missing returns set to the mandi's mean return; Ledoit-Wolf shrinkage; PMFG with  
d_ij = sqrt(2(1-rho_ij)) and edge weight 1/d_ij), applied to the cleaned panel.  
Usage: python 02_network.py [--shrinkage 0.15] [--tag alpha015]   (robustness variants)

In [ ]:
%run 02_network.py --pairwise --tag pairwise

### Step 16. `02b_missing_data_check.py` 

*Feeds:* Section 4.1 (missing-data check)

Sensitivity of correlations and centrality to the treatment of missing price changes (Section 3.4.1).  
  
The baseline network sets missing log price changes to the mandi's mean change before estimating the Ledoit-Wolf  
covariance matrix. With about 39 percent of mandi-days missing, this shrinks covariances toward zero for thinly  
reporting mandis. Two alternatives avoid imputation altogether:  
  (a) pairwise-complete sample correlations of the same changes, computed on the dates both mandis report  
      (at least 60 common changes);  
  (b) as (a), but using only one-day changes (consecutive trading days), so that every change spans the same horizon.  
For each, the PMFG is rebuilt and eigenvector centrality recomputed, and compared with the baseline: Spearman rank  
correlation of eigenvector centrality, overlap of the ten most central mandis, the rank of the Nashik-belt (C0)  
mandis, and the mean off-diagonal correlation.

In [ ]:
%run 02b_missing_data_check.py

### Step 17. `10_asymmetry_simam_partitions.py` 

*Feeds:* Sections V.C–D; Table 6; Appendix Table B3

(a) Band-TAR asymmetry, (b) SIMAM density and regime differences, (c) robustness of the  
adjustment pattern to the market partition (alternative network partitions and state indices).  
  
(a) Three regimes on the signed ECT_{t-1}: below (ECT <= g_L < 0), band, above (ECT > g_U > 0).  
    g_L and g_U are searched over quantiles of the negative and positive ECT values; each outer  
    regime holds >= 10% and the band >= 10% of observations. The criterion is the SSR of the dECT  
    equation (same regressors as the TVECM). The original notebook did not force the band to  
    contain zero and its upper regime always sat at the 15% trimming bound (219 obs for all pairs).  
    H0: theta_below = theta_above, Newey-West (10 lags).  
    Sign convention: ECT = C_j - a - b C_i. "Above" = destination community j is expensive  
    relative to the long-run relation with i.  
(b) Pair outcome: theta_2 - theta_1 (negative = faster adjustment in the outer regime).  
    Regressor: mean SIMAM share of the two communities. Inference: exact permutation test that  
    permutes community-level SIMAM shares across communities (all K! permutations).  
(c) The TVECM (point estimates and HAC regime-equality tests, no bootstrap) is re-estimated on  
    indices built from alternative partitions.

In [ ]:
%run 10_asymmetry_simam_partitions.py
show('table_bandtar')

### Step 18. `13_economic_interpretation.py` 

*Feeds:* Section VII; Appendix Tables B4–B5

Economic interpretation: thresholds in rupees, thresholds and distance, production-belt rainfall.  
  
(a) Implied transaction-cost band in Rs/quintal: (exp(gamma) - 1) x mean price of the pair's two indices.  
(b) Spatial arbitrage predicts wider no-trade bands between more distant markets. We regress gamma (and  
    theta_2 - theta_1) on log great-circle distance between the arrival-weighted centroids of the two  
    communities (verified coordinates), and on the number of direct PMFG edges between them.  
    N = 28 pairs that share communities, so inference uses the exact community-permutation test  
    (relabelling communities' locations), as for SIMAMs.  
(c) Supply-side rainfall: rainfall stress in the Nashik production belt (community C0 exposure) is a  
    national supply shock. Monthly local projections of the system price index (arrival-weighted,  
    240 mandis) on C0 deficit measures with calendar-month effects and Newey-West (3) errors.

In [ ]:
%run 13_economic_interpretation.py
show('table_thresholds_distance')
show('table_production_rain_lp')

### Step 19. `14_community_iv.py` 

*Feeds:* Section VI.C

Noise-robust check of gap-dependent adjustment between community pairs (Section VI.C).  
  
Weekly means of the daily community price indices (04_indices_cointegration.py). For every pair of communities the  
gap g_t is demeaned and the pooled model  
  
    Delta g_t = -(theta + pi |q_t|) g_{t-1} + e_t  
  
is estimated over all 28 pairs. Conventional estimator: q = g_{t-1}, OLS. Noise-robust estimator: q = g_{t-k}  
(k = 2, 3) as an exogenous threshold variable, instruments (g_{t-k}, |g_{t-k}| g_{t-k}) (Caner and Hansen 2004).  
pi > 0 means large gaps close faster. Inference resamples community pairs (999 bootstrap draws).  
Speeds by quintile of the gap size are also reported.

In [ ]:
%run 14_community_iv.py

### Step 20. `15_monte_carlo.py` 

*Feeds:* Section 5.3 (Monte Carlo of the estimators)

Monte Carlo: what can the estimators of Section 5.3 distinguish? (Online Appendix Table B8)  
  
Each replication simulates 28 independent community-pair gaps at weekly frequency, T = 210 weeks, calibrated to the  
weekly community gaps used in 14_community_iv.py: from the autocovariances of the observed gaps, the median AR(1)  
coefficient of the persistent gap is 0.825 (speed 0.175 a week), its variance 0.0141, and the variance of transitory  
noise 0.0020 (about 12 percent of the observed variance). The observed gap is q_t = g_t + v_t.  
  
True dynamics (a two-regime band model, the model the paper estimates):  
    Delta g_t = -theta1 g_{t-1} 1(|g_{t-1}| <= 0.15) - theta2 g_{t-1} 1(|g_{t-1}| > 0.15) + e_t,  
with theta1 = 0.175 and theta2 = 0.175 (linear), 0.35 (large gaps close twice as fast), 0.0875 (half as fast:  
fragmentation) or 0.03 (large gaps barely close). About a fifth of weeks lie in the outer regime, as in the data.  
e_t is Gaussian with the variance that gives var(g) = 0.0141 in the linear case.  
Transitory noise v: none, Gaussian, or Student-t with 5 or 3 degrees of freedom, all with variance 0.0020.  
  
Estimators applied to each simulated panel (as in the paper):  
  conv : pooled OLS of Delta q_t on q_{t-1} and |q_{t-1}| q_{t-1}; pi > 0 means large gaps close faster;  
  iv   : the same with q_{t-3} as threshold variable and (q_{t-3}, |q_{t-3}| q_{t-3}) as instruments  
         (14_community_iv.py); inference for both by resampling pairs (199 draws), two-sided 5 percent tests;  
  tvecm: per pair, a two-regime threshold model on |q_{t-1}| (threshold searched over its 15th-85th percentiles),  
         counting pairs with a faster outer regime, and pairs with a significantly faster or slower outer regime  
         (heteroskedasticity-robust t test, 5 percent).

In [ ]:
%run 15_monte_carlo.py
show('table_monte_carlo')

### Step 21. `11_figures.py` 

*Feeds:* Figures 1–3; Appendix Figure B1

Figures for the paper (static, print-ready PNG at 300 dpi).

In [ ]:
%run 11_figures.py

### Step 22. `12_paper_numbers.py` 

*Feeds:* Every number quoted in the text

Format every number quoted in the manuscript from output/results.json.  
  
Writes output/paper_numbers.json: {"N": {...scalar strings...}, "T": {...tables as row lists...}}.  
The manuscript builder (paper/build_paper.js) reads only this file, so text and tables cannot  
drift from the estimates.

In [ ]:
%run 12_paper_numbers.py

### Figures

In [ ]:
for f in ["fig1_communities_map.png", "fig2_tvecm_regimes.png", "fig3_rain_local_projections.png", "fig4_rolling_fiedler.png"]:
    display(Image(filename=os.path.join("..", "output", "figures", f), width=700))